## telnetで手動でやったのと同じ動作を python で書いてみる

### 単語の意味
- request
  - 要求、請求、要請
- response
  - 反応、応答、答え

In [ ]:
import requests

# GETで行う場合
response = requests.get( 'http://mdqu.rcis.jp/kibanp-taskA.php', params={'num':8763} )
print(response.url)  # どのようなURLで呼び出したかを表示
print('---- 応答結果 ----')
print(response.text)

# POSTで行う場合
response2 = requests.post( 'http://mdqu.rcis.jp/kibanp-taskA.php', data={'num':8762} )
print(response2.url)  # どのようなURLで呼び出したかを表示
print('---- 応答結果2 ----')
print(response2.text)

## 外部サイトからリアルタイムのデータをもらって処理する例
Open-Meteo
https://open-meteo.com/
というサイトで、世界のあらゆる場所の天気情報を取得できる[API](https://ja.wikipedia.org/wiki/%E3%82%A2%E3%83%97%E3%83%AA%E3%82%B1%E3%83%BC%E3%82%B7%E3%83%A7%E3%83%B3%E3%83%97%E3%83%AD%E3%82%B0%E3%83%A9%E3%83%9F%E3%83%B3%E3%82%B0%E3%82%A4%E3%83%B3%E3%82%BF%E3%83%95%E3%82%A7%E3%83%BC%E3%82%B9)が無料で公開されています。

例えば、熊本大学付近の今の気温であれば  
https://api.open-meteo.com/v1/forecast?latitude=32.8133&longitude=130.7281&current=temperature_2m  
というURLで得ることができます。([JSON](https://ja.wikipedia.org/wiki/JSON)という形式のデータが返ってきます）


**注意**： 以下のコードには３箇所に**間違い**があります。ちゃんと実行できるように直してみてください。

In [ ]:
# 必要なライブラリをインポート
import requests
import matplotlib.pyplot as plt
import japanize_matplotlib
from mpl_toolkits.mplot3d import Axes3D # 3Dプロット用

In [ ]:
# 九州7県の県庁所在地の緯度・経度データ
kyushu_capitals = {
    'Fukuoka':   {'lat': 33.5904, 'lon': 130.4017},
    'Saga':      {'lat': 33.2494, 'lon': 130.2974},
    'Nagasaki':  {'lat': 32.7450, 'lon': 129.8739},
    'Kumamoto':  {'lat': 32.7900, 'lon': 130.7420},
    'Oita':      {'lat': 33.2381, 'lon': 131.6119},
    'Miyazaki':  {'lat': 31.9110, 'lon': 131.4240},
    'Kagoshima': {'lat': 31.5600, 'lon': 130.5580}
}

weather_info = []  # 結果を保存するためのリスト
BASE_URL = 'https://api.open-meteo.com/v1/forecast'

print('--- Open-Meteo APIから気温データを取得 ---')
for city, coords in kyushu_capitals.items():
    params = {
        'latitude':  coords['lat'],
        'longitude': coords['lon'],
        'current': 'temperature_2m'
    }

    try:
        # APIリクエストの実行
        response = requests.get(BASE_URL, params=params)
        print(f'URL: {response.url}')  # APIをどのように呼び出したかを表示
        response.raise_for_status()  # エラーの場合は例外を発生、exceptに飛ぶ
        data = response.json()  # JSON形式の文字データをリストなどに変換

        # 気温データを抽出
        tempe = data['current']['temperature_2m']
        
        # データをリストに追加
        weather_info.append({
            'City': city,
            'Latitude': coords['lat'],
            'Longitude': coords['lon']
            'Temperature': tempe
        })

        print(f' ✅ {city}: {tempe}°C を取得しました。')

    except requests.exceptions.RequestException as e:
        print(f'error: {e}')

# リストをDataFrameに変換
df = pandas.DataFrame(weather_info)

print('\n--- 取得結果 ---)
print(df)

In [ ]:
# 3Dプロットの設定
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# --- データを色付きでプロット ---
scatter = ax.scatter(
    df['Longitude'], 
    df['Latitude'], 
    df['Temperature'], 
    c=df['Temperature'], 
    cmap='viridis', 
    s=300,        
    depthshade=True
)

# データフレームの各行（各都市）に対して処理を行う
for i in range(len(df)):
    lon = df['Longitude'][i]
    lat = df['Latitude'][i]
    tempe = df['Temperature'][i]
    city = df['City'][i]

    # データ点 (lon, lat, tempe) から 底面 (lon, lat, Z_min) へ線 (Z方向に垂直) を引く
    Z_min = df['Temperature'].min() - 5  # Z軸の適当な最小値を設定
    ax.plot(
        [lon, lon],  # X座標は変化しない
        [lat, lat],  # Y座標は変化しない
        [tempe, Z_min], # Z座標が (temp) から (Z_min) まで変化
        color='gray', 
        linestyle='--', 
        linewidth=2, 
        alpha=0.6
    )
    
    # Z_min の位置にラベルを付けることで、地面付近に都市名を表示
    ax.text(
        lon, 
        lat, 
        Z_min, 
        f'{city}\n({tempe:.1f}°C)', # 2行で表示
        size=9, 
        color='black',
        ha='center', # 水平方向を中央揃え
        va='top'     # 垂直方向を上揃え（線の下に表示されるように）
    )

# --- ラベルとタイトルの設定 ---
ax.set_title('九州主要都市の現在の気温 3Dプロット', fontsize=16)
ax.set_xlabel('経度 (Longitude)', fontsize=12)
ax.set_ylabel('緯度 (Latitude)', fontsize=12) 
ax.set_zlabel('現在の気温 (°C)', fontsize=12)

# Z軸の範囲を、投影線が収まるように調整
ax.set_zlim(Z_min, df['Temperature'].max() + 5) 

cbar = fig.colorbar(scatter, ax=ax, pad=0.1, shrink=0.5)  # カラーバー
cbar.set_label('気温 (°C)', rotation=90, labelpad=10)

ax.view_init(elev=20, azim=-75)  # 視点の調整 (仰角20度, 方位角-75度)
plt.show()